In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

print("Task 3 Libraries imported successfully!")

Task 3 Libraries imported successfully!


In [3]:
# 1. Load the dataset cleanly from scikit-learn
data = fetch_california_housing(as_frame=True)

# 2. Combine features and target into one DataFrame (with the bracket typo fixed!)
df = pd.concat([data.data, data.target.rename("HousePrice")], axis=1)

# 3. Separate features (X) and target variable (y)
X = df.drop("HousePrice", axis=1)
y = df["HousePrice"]

print("Dataset loaded and split into X and y successfully!")
df.head()

Dataset loaded and split into X and y successfully!


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,HousePrice
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [2]:
# 1. Load the dataset cleanly from scikit-learn
data = fetch_california_housing(as_frame=True)

# 2. Combine features and target into one DataFrame (fixed syntax typo)
df = pd.concat([data.data, data.target.rename("HousePrice")], axis=1)

# 3. Separate features (X) and target variable (y)
X = df.drop("HousePrice", axis=1)
y = df["HousePrice"]

print("Dataset loaded and split into X and y successfully!")
df.head()

Dataset loaded and split into X and y successfully!


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,HousePrice
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [4]:
# Initialize and apply the Standard Scaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split the data into an 80% training set and 20% validation/test set
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print("Step 3 & 4 Completed: Data successfully scaled and split!")

Step 3 & 4 Completed: Data successfully scaled and split!


In [6]:
# Import the updated RMSE function directly
from sklearn.metrics import root_mean_squared_error

# Train a baseline Decision Tree with no constraints [cite: 358, 359]
tree = DecisionTreeRegressor(random_state=42) # [cite: 359]
tree.fit(X_train, y_train) # [cite: 360]

# Generate predictions for both datasets [cite: 361, 362]
train_pred = tree.predict(X_train) # [cite: 361]
test_pred = tree.predict(X_test) # [cite: 362]

# Calculate Root Mean Squared Error (RMSE) using the updated scikit-learn functions
train_rmse = root_mean_squared_error(y_train, train_pred)
test_rmse = root_mean_squared_error(y_test, test_pred)

print(f"Training RMSE (Perfect Memorization): {train_rmse:.4f}")
print(f"Testing RMSE (Unseen Data Performance): {test_rmse:.4f}")
print(f"Performance Gap: {test_rmse - train_rmse:.4f} -> This massive gap confirms Overfitting! ")

Training RMSE (Perfect Memorization): 0.0000
Testing RMSE (Unseen Data Performance): 0.7030
Performance Gap: 0.7030 -> This massive gap confirms Overfitting! 


In [7]:
# Perform 5-Fold Cross-Validation to evaluate stable baseline error
cv_scores = cross_val_score(
    tree, X_scaled, y,
    scoring="neg_root_mean_squared_error",
    cv=5
)

# Convert negative scores back to a positive average RMSE
cv_rmse = -cv_scores.mean()
print(f"Realistic 5-Fold Cross-Validated RMSE: {cv_rmse:.4f}")

Realistic 5-Fold Cross-Validated RMSE: 0.8957


In [8]:
# Define hyperparameter structural constraints to test
param_grid = {
    "max_depth": [3, 5, 7, 10],
    "min_samples_split": [2, 5, 10, 20]
}

# Set up the Grid Search cross-validation pipeline
grid = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid,
    scoring="neg_root_mean_squared_error",
    cv=5
)

print("Tuning hyperparameters across the grid... (This will take 2-3 seconds)")
grid.fit(X_train, y_train)

print("\nBest Parameters Found:", grid.best_params_)

Tuning hyperparameters across the grid... (This will take 2-3 seconds)

Best Parameters Found: {'max_depth': 10, 'min_samples_split': 20}


In [9]:
# 1. Extract the optimized model configurations
best_tree = grid.best_estimator_

# 2. Predict on the unseen testing data
y_pred = best_tree.predict(X_test)

# 3. Compute final validation metrics using the updated scikit-learn functions
from sklearn.metrics import root_mean_squared_error, r2_score
tuned_rmse = root_mean_squared_error(y_test, y_pred)
tuned_r2 = r2_score(y_test, y_pred)

print(f"Optimized Tuned Tree RMSE: {tuned_rmse:.4f}")
print(f"Optimized Tuned Tree R2 Score: {tuned_r2:.4f}")

Optimized Tuned Tree RMSE: 0.6347
Optimized Tuned Tree R2 Score: 0.6926


In [10]:
# Construct a master comparison summary incorporating Task 2 static baseline metrics
results_summary = {
    "Model": ["Linear Regression (Task 2)", "Ridge Regression (Task 2)", "Tuned Decision Tree (Task 3)"],
    "RMSE (Lower = Better)": [0.7455, 0.7455, tuned_rmse],
    "R2 Score (Higher = Better)": [0.5758, 0.5758, tuned_r2]
}

# Render as a clean pandas DataFrame table
summary_df = pd.DataFrame(results_summary)
print("\n================= FINAL PROJECT METRICS SUMMARY =================")
print(summary_df.to_string(index=False))


================= FINAL PROJECT METRICS SUMMARY =================
                       Model  RMSE (Lower = Better)  R2 Score (Higher = Better)
  Linear Regression (Task 2)               0.745500                    0.575800
   Ridge Regression (Task 2)               0.745500                    0.575800
Tuned Decision Tree (Task 3)               0.634665                    0.692616
